# Candlestick pattern scanner (dYdX catalog)

Scans the collector's `custom_dydx_minute_bar` catalog data for **all TA-Lib candlestick
patterns** (via `pandas_ta`'s `cdl_pattern(name="all")`, 60+ patterns including hammer/doji
variants) across **multiple timeframes** at once (built by resampling the native 1-minute bars --
no extra data collection needed), filtered against a condition (e.g. above/below EMA50), and
gives you a clickable list that jumps a candlestick chart to each match at the right timeframe.

**Data coverage caveat:** the collector's catalog currently only has `custom_dydx_minute_bar`
data for ~2026-06-30 to 2026-07-02 (a few days), not a full year -- the collector wasn't
running continuously through the requested window. `LOOKBACK_DAYS` below is a request, not a
guarantee; the notebook prints what's actually available and clips to it. Higher timeframes
(4h/1D) will only have a handful of bars given that short a history -- run the collector longer
to build up more. Run the collector longer to build up more history for future scans.

**Deps:** `pandas_ta` (all-patterns path needs the real `TA-Lib` C bindings, not the pure-python
fallback) isn't in this repo's locked `uv.lock` -- it's notebook-only tooling, not added to the
main project dependencies. The install cell below is idempotent (skips if already installed) and
pins `numba` to a version that won't downgrade this repo's numpy, which `nautilus_trader`'s
compiled extensions depend on. If `uv sync` ever wipes them, just re-run the install cell.


In [ ]:
try:
    import pandas_ta  # noqa: F401
    import talib  # noqa: F401
except ImportError:
    # ponytail: notebook-only deps, not in pyproject/uv.lock -- see markdown cell above.
    # numba is pinned to a recent version explicitly: an unconstrained resolve can drag in
    # an old numba that downgrades numpy, which breaks nautilus_trader's compiled extensions.
    %pip install -q --pre pandas_ta
    %pip install -q TA-Lib "numba>=0.61.2"
    import pandas_ta  # noqa: F401
    import talib  # noqa: F401
print("pandas_ta + TA-Lib ready")


In [ ]:
import glob

import ipywidgets as widgets
import pandas as pd
import pandas_ta as ta
import plotly.graph_objects as go
from IPython.display import display

from nautilus_trader.persistence.catalog import ParquetDataCatalog

pd.set_option("display.max_rows", 200)


## Parameters

Pick an instrument, a lookback window, the EMA length, and which side of the EMA counts as a
match. `PATTERN_FILTER=None` scans every TA-Lib pattern; set it to a substring (e.g. `"HAMMER"`
or `"DOJI"`) to narrow the scan. `TIMEFRAMES` are pandas resample rules applied to the native
1-minute bars -- add/remove freely.


In [ ]:
CATALOG_PATH = "../catalog"  # relative to this notebook's directory

catalog = ParquetDataCatalog(CATALOG_PATH)
minute_bar_instruments = sorted(
    p.rsplit("/", 1)[-1]
    for p in glob.glob(f"{CATALOG_PATH}/data/custom_dydx_minute_bar/*")
)
print(f"{len(minute_bar_instruments)} instruments have minute-bar data")
minute_bar_instruments[:10]


In [ ]:
INSTRUMENT_ID = "BTC-USD-PERP.DYDX"
LOOKBACK_DAYS = 365  # requested window; clipped to what's actually in the catalog (see above)
EMA_LEN = 50
CONDITION = "above"  # "above", "below", or "any" -- close vs EMA{EMA_LEN}
PATTERN_FILTER = None  # e.g. "HAMMER" / "DOJI" / "ENGULFING", or None for every pattern
TIMEFRAMES = ["1min", "5min", "15min", "1h", "4h", "1D"]  # pandas resample rules


## Load 1-minute bars

Reads `custom_dydx_minute_bar` parquet directly with pandas rather than through Nautilus's
typed catalog API -- the `DydxMinuteBar` custom `Data` class those files were written with isn't
present on this branch, and the stored columns are already plain floats (no fixed-point
Price/Quantity decoding needed), so a raw read is both simpler and correct here.


In [ ]:
def load_minute_bars(instrument_id: str, lookback_days: int) -> pd.DataFrame:
    path = f"{CATALOG_PATH}/data/custom_dydx_minute_bar/{instrument_id}"
    df = pd.read_parquet(path)
    df = df.drop_duplicates(subset="ts_event").sort_values("ts_event")
    df["timestamp"] = pd.to_datetime(df["ts_event"], unit="ns", utc=True)
    df = df.set_index("timestamp")

    available_start, available_end = df.index.min(), df.index.max()
    start = max(available_start, available_end - pd.Timedelta(days=lookback_days))
    df = df.loc[start:available_end]

    print(f"{instrument_id}: catalog has {available_start} -> {available_end}")
    print(f"requested {lookback_days}d, using {df.index.min()} -> {df.index.max()} ({len(df)} bars)")
    return df


bars_1m = load_minute_bars(INSTRUMENT_ID, LOOKBACK_DAYS)
bars_1m[["open", "high", "low", "close", "volume"]].tail()


## Resample to every timeframe

Same OHLCV source, just re-aggregated per `TIMEFRAMES` -- reuses `bars_1m` from the cell above
instead of pulling more data.


In [ ]:
def resample_bars(bars_1m: pd.DataFrame, rule: str) -> pd.DataFrame:
    ohlc = bars_1m[["open", "high", "low", "close"]].resample(rule).agg(
        {"open": "first", "high": "max", "low": "min", "close": "last"}
    )
    volume = bars_1m["volume"].resample(rule).sum()
    out = ohlc.join(volume).dropna(subset=["open", "high", "low", "close"])
    return out


bars_by_tf = {tf: resample_bars(bars_1m, tf) for tf in TIMEFRAMES}
for tf, df in bars_by_tf.items():
    print(f"{tf:>5}: {len(df)} bars")


## Compute EMA + scan every TA-Lib candlestick pattern, per timeframe

`cdl_pattern(name="all")` runs every TA-Lib `CDL*` recognizer over each timeframe's OHLC series.
Each output column is a signed int per bar: `+100`/`+200` = bullish signal, `-100`/`-200` =
bearish, `0` = no signal. Matches from every timeframe are combined into one table, tagged by
`timeframe`.


In [ ]:
def scan_patterns(bars: pd.DataFrame, timeframe: str) -> pd.DataFrame:
    bars = bars.copy()
    bars["ema"] = bars["close"].ewm(span=EMA_LEN, adjust=False).mean()

    patterns = bars.ta.cdl_pattern(name="all")
    patterns.columns = [c.replace("CDL_", "") for c in patterns.columns]

    hits = patterns[(patterns != 0).any(axis=1)].stack()
    hits = hits[hits != 0]
    hits.index.names = ["timestamp", "pattern"]

    matches = hits.rename("signal").reset_index()
    matches["direction"] = matches["signal"].apply(lambda v: "bullish" if v > 0 else "bearish")
    matches = matches.merge(
        bars[["open", "high", "low", "close", "ema"]], left_on="timestamp", right_index=True
    )
    matches["timeframe"] = timeframe
    return matches


all_matches = pd.concat(
    [scan_patterns(df, tf) for tf, df in bars_by_tf.items() if len(df) > 0], ignore_index=True
)

if PATTERN_FILTER:
    all_matches = all_matches[all_matches["pattern"].str.contains(PATTERN_FILTER, case=False)]

if CONDITION == "above":
    all_matches = all_matches[all_matches["close"] > all_matches["ema"]]
elif CONDITION == "below":
    all_matches = all_matches[all_matches["close"] < all_matches["ema"]]
# CONDITION == "any" -> no filter

all_matches = all_matches.sort_values("timestamp", ascending=False).reset_index(drop=True)
print(f"{len(all_matches)} matches across {len(TIMEFRAMES)} timeframes")
all_matches[["timestamp", "timeframe", "pattern", "direction", "close", "ema"]]


## Clickable list -> jump to chart

Pick a match from the dropdown (timeframe shown in the label); the candlestick chart below
redraws centered on that bar **at that match's timeframe**, with EMA{EMA_LEN} overlaid and the
matched bar marked.


In [ ]:
WINDOW_BARS = 40  # bars shown on each side of the matched bar

chart_out = widgets.Output()

options = [
    (
        f"{row.timestamp:%Y-%m-%d %H:%M} UTC  [{row.timeframe}]  {row.pattern} ({row.direction})",
        i,
    )
    for i, row in all_matches.iterrows()
]
picker = widgets.Dropdown(options=options, description="Match:", layout=widgets.Layout(width="560px"))


def render(match_idx: int) -> None:
    row = all_matches.loc[match_idx]
    bars = bars_by_tf[row.timeframe]
    ema = bars["close"].ewm(span=EMA_LEN, adjust=False).mean()

    pos = bars.index.get_indexer([row.timestamp])[0]
    window = bars.iloc[max(0, pos - WINDOW_BARS) : pos + WINDOW_BARS + 1]
    window_ema = ema.loc[window.index]

    fig = go.Figure(
        data=[
            go.Candlestick(
                x=window.index,
                open=window["open"],
                high=window["high"],
                low=window["low"],
                close=window["close"],
                name=INSTRUMENT_ID,
            ),
            go.Scatter(x=window.index, y=window_ema, name=f"EMA{EMA_LEN}", line=dict(width=1)),
        ]
    )
    fig.add_vline(x=row.timestamp, line_dash="dot", line_color="gray")
    fig.add_annotation(
        x=row.timestamp, y=window["high"].max(), text=f"{row.pattern} ({row.direction})",
        showarrow=True, arrowhead=1, yshift=10,
    )
    fig.update_layout(
        title=f"{INSTRUMENT_ID} [{row.timeframe}] around {row.timestamp:%Y-%m-%d %H:%M} UTC",
        xaxis_rangeslider_visible=False,
        height=450,
    )
    chart_out.clear_output(wait=True)
    with chart_out:
        fig.show()


def on_change(change):
    if change["name"] == "value" and change["new"] is not None:
        render(change["new"])


picker.observe(on_change, names="value")
display(picker, chart_out)
if options:
    render(options[0][1])
